# ⚡ YOLO Vision Studio: Cloud GPU Training on Kaggle
This notebook is generated for training YOLO models (YOLO11, YOLOv8, YOLOv9, YOLOv10) with **Dual NVIDIA T4 (32GB VRAM) / P100 GPU** acceleration on Kaggle.

### ⚙️ Quick Kaggle Settings Checklist:
1. **Accelerator**: Right sidebar -> **Settings** -> **Accelerator** -> Select **GPU T4 x 2** (or GPU P100).
2. **Internet**: Right sidebar -> **Settings** -> **Internet** -> Toggle **ON** (required to download pre-trained weights & packages).
3. **Dataset**: Ensure your YOLO dataset (e.g., `cctv-1`) is attached under **Data** -> **+ Add Input**.

In [ ]:
# 1. Check Hardware & Install Dependencies
!nvidia-smi
!pip install -q --upgrade ultralytics pyyaml opencv-python Pillow matplotlib

In [ ]:
import os
import sys
import glob
import yaml
import shutil
from pathlib import Path
import torch
from ultralytics import YOLO

print(f"CUDA Available: {torch.cuda.is_available()}")
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"Detected GPUs: {gpu_count}")
for i in range(gpu_count):
    print(f"  [{i}] {torch.cuda.get_device_name(i)}")

# Determine multi-GPU or single-GPU
device = [0, 1] if gpu_count >= 2 else (0 if gpu_count == 1 else 'cpu')
print(f"Active Training Device: {device}")

In [ ]:
# 2. Locate and Prepare Dataset YAML
input_dir = Path("/kaggle/input")
dataset_dir = None

for root, dirs, files in os.walk(input_dir):
    if "data.yaml" in files or "dataset.yaml" in files:
        dataset_dir = Path(root)
        break

if not dataset_dir:
    raise FileNotFoundError("data.yaml not found in /kaggle/input/. Please attach your dataset via '+ Add Input'.")

yaml_path = dataset_dir / ("data.yaml" if (dataset_dir / "data.yaml").exists() else "dataset.yaml")
print(f"Found dataset config at: {yaml_path}")

with open(yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

# Patch root path to /kaggle/input dataset directory
data_config["path"] = str(dataset_dir)
kaggle_yaml = Path("/kaggle/working/data_kaggle.yaml")
with open(kaggle_yaml, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"Prepared patched data config at: {kaggle_yaml}")

In [ ]:
# 3. Configure Training Parameters
MODEL_NAME = "yolo11n.pt"    # Options: yolo11n.pt, yolo11s.pt, yolo11m.pt, yolov8n.pt, etc.
EPOCHS = 100
BATCH_SIZE = 32             # Use 32 or 64 with 2x T4 GPUs
IMGSZ = 640
OPTIMIZER = "AdamW"         # Options: AdamW, SGD, Adam, auto
LR0 = 0.001
PATIENCE = 20
PROJECT = "/kaggle/working/yolo_runs"
EXP_NAME = "train_exp"

print(f"Initializing YOLO model: {MODEL_NAME}...")
model = YOLO(MODEL_NAME)

In [ ]:
# 4. Launch Training
results = model.train(
    data=str(kaggle_yaml),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    optimizer=OPTIMIZER,
    lr0=LR0,
    patience=PATIENCE,
    device=device,
    project=PROJECT,
    name=EXP_NAME,
    exist_ok=True,
    plots=True,
    save=True,
    verbose=True
)
print("Training Complete!")

In [ ]:
# 5. Display Validation Metrics and Curves
from IPython.display import Image, display

run_dir = Path(PROJECT) / EXP_NAME

for plot_name in ["results.png", "confusion_matrix.png", "PR_curve.png", "val_batch0_pred.jpg"]:
    plot_file = run_dir / plot_name
    if plot_file.exists():
        print(f"\n--- {plot_name} ---")
        display(Image(filename=str(plot_file)))

In [ ]:
# 6. Package Weights & Output for 1-Click Download
best_weight = run_dir / "weights" / "best.pt"
print(f"Best model saved at: {best_weight}")

# Create downloadable zip archive in /kaggle/working
archive_name = "/kaggle/working/yolo_trained_model"
shutil.make_archive(archive_name, 'zip', root_dir=str(run_dir))
print(f"Created downloadable zip archive: {archive_name}.zip")